# Laboratorio 8 — Datos abiertos y APIs

María Paula Angulo Collantes

Danna Gabriela Castro

Dataset: *Servicio de conectividad a internet en instituciones educativas del departamento de Norte de Santander* (datos.gov.co)

## 1. Dataset seleccionado y registro de la fuente

| Campo | Registro |
|---|---|
| Tema | Educación |
| Portal | Datos Abiertos Colombia (datos.gov.co) |
| ID del dataset | `px6y-fznz` |
| Entidad publicadora | [ confirmar con el metadato del portal: ver celda siguiente ] |
| URL (API) | https://www.datos.gov.co/api/v3/views/px6y-fznz/query.json |
| Fecha de consulta | 15 de octubre de 2026 |

In [1]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime

ID = "px6y-fznz"
URL = f"https://www.datos.gov.co/api/v3/views/{ID}/query.json"
fecha_consulta = datetime.now().strftime("%d/%m/%Y %H:%M")
print("Fecha de consulta:", fecha_consulta)

# Metadato del portal: nombre y entidad responsable del dataset
try:
    meta = requests.get(f"https://www.datos.gov.co/api/views/{ID}.json", timeout=30).json()
    print("Nombre:", meta.get("name"))
    print("Entidad / atribución:", meta.get("attribution"))
    print("Publicado por:", meta.get("owner", {}).get("displayName"))
except Exception as e:
    print("No se pudo leer el metadato:", e)

Fecha de consulta: 15/10/2026 17:34
Nombre: SERVICIO DE CONECTIVIDAD A INTERNET EN INSTITUCIONES EDUCATIVAS DEL DEPARTAMENTO NORTE DE SANTANDER
Entidad / atribución: Gobernación de Norte de Santander, Norte de Santander
Publicado por: datosabiertos@nortedesantander.gov.co


## 2. Carga con `requests` y validación inicial de estructura

In [2]:
payload = {
    "query": "SELECT * LIMIT 1000",
    "page": {"pageSize": 1000, "pageNumber": 1},
}
# Si el portal exige token: headers = {"X-App-Token": "TU_TOKEN"}

try:
    r = requests.post(URL, json=payload, timeout=60)
    print("Status code:", r.status_code)
    r.raise_for_status()
    data = r.json()
    print("Tipo de la respuesta:", type(data))
    print("Llaves:", list(data.keys()) if isinstance(data, dict) else f"lista de {len(data)} registros")
except Exception as e:
    print("Falló la API:", e)
    data = None

Status code: 200
Tipo de la respuesta: <class 'list'>
Llaves: lista de 254 registros


In [3]:
if data is not None:
    registros = data.get("data", data) if isinstance(data, dict) else data
    df = pd.DataFrame(registros)
    origen = "API"
else:
    # Respaldo: el CSV descargado del portal (misma fuente)
    df = pd.read_csv("SERVICIO_DE_CONECTIVIDAD_A_INTERNET_EN_INSTITUCIONES_EDUCATIVAS_DEL_DEPARTAMENTO_NORTE_DE_SANTANDER_20261015.csv")
    origen = "CSV local"

print("Origen de los datos:", origen)
print("Dimensiones:", df.shape)
df.head()

Origen de los datos: API
Dimensiones: (254, 17)


,id,departamento,municipio,codigo_dane_sede,nombre_sede_educativa,matr_cula_a_beneficiar_por,zona_urbana_rural,tecnolog_a_ltima_milla_cobre,ancho_de_banda_mbps_capacidad,conetividad_para_estudiantes,meses_de_servicio,fecha_inicio_servicio,fecha_fin_servicio,:id,:version,:created_at,:updated_at
0,1,NORTE DE SANTANDER,ABREGO,154003000262,SEDE VARONES,475,URBANA,Radio,20Mbps,CONECTIVIDAD EN SEDE,4,01 DE JULIO DE 2022,30 DE NOVIEMBRE DE 2022,row-xsmr_uvvv_7n2z,rv-u3g9.vxs5-s66x,2022-06-16T18:36:12.334Z,2022-06-16T18:36:12.334Z
1,2,NORTE DE SANTANDER,ABREGO,154003000921,SEDE INTEGRADA SAN RAFAEL,492,URBANA,Radio,20Mbps,CONECTIVIDAD EN SEDE,4,01 DE JULIO DE 2022,30 DE NOVIEMBRE DE 2022,row-cubh.4tw6-qprs,rv-sdkf.y57t.jcqt,2022-06-16T18:36:12.334Z,2022-06-16T18:36:12.334Z
2,3,NORTE DE SANTANDER,ABREGO,154003000840,SEDE SAN ANTONIO,270,URBANA,Radio,20Mbps,CONECTIVIDAD EN SEDE,4,01 DE JULIO DE 2022,30 DE NOVIEMBRE DE 2022,row-vx9f.mzbv-rkym,rv-b26g.fyfe_7vpa,2022-06-16T18:36:12.334Z,2022-06-16T18:36:12.334Z
3,4,NORTE DE SANTANDER,ABREGO,154003001668,SEDE CARLOS JULIO TORRADO PEÑARANDA,1600,URBANA,Radio,20Mbps,CONECTIVIDAD EN SEDE,4,01 DE JULIO DE 2022,30 DE NOVIEMBRE DE 2022,row-u559.dr6h.a5hn,rv-qe7y~8nma.xzbx,2022-06-16T18:36:12.334Z,2022-06-16T18:36:12.334Z
4,5,NORTE DE SANTANDER,ABREGO,154003000823,SEDE SANTA BARBARA,1075,URBANA,Radio,20Mbps,CONECTIVIDAD EN SEDE,4,01 DE JULIO DE 2022,30 DE NOVIEMBRE DE 2022,row-cfhq.u8em_rc66,rv-q96k-24jk.gd4b,2022-06-16T18:36:12.334Z,2022-06-16T18:36:12.334Z


In [4]:
# Validación inicial de estructura
df.info()
print()
print("Nulos por columna:")
print(df.isna().sum())
print()
print("Filas duplicadas:", df.duplicated().sum())
print()
print("Valores distintos por columna:")
print(df.nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 254 entries, 0 to 253
Data columns (total 17 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   id                             254 non-null    object
 1   departamento                   254 non-null    object
 2   municipio                      254 non-null    object
 3   codigo_dane_sede               254 non-null    object
 4   nombre_sede_educativa          254 non-null    object
 5   matr_cula_a_beneficiar_por     254 non-null    object
 6   zona_urbana_rural              254 non-null    object
 7   tecnolog_a_ltima_milla_cobre   254 non-null    object
 8   ancho_de_banda_mbps_capacidad  254 non-null    object
 9   conetividad_para_estudiantes   254 non-null    object
 10  meses_de_servicio              254 non-null    object
 11  fecha_inicio_servicio          254 non-null    object
 12  fecha_fin_servicio             254 non-null    object
 13  :id  

In [5]:
# Los nombres de columna cambian entre la API y el CSV: se ubican por palabra clave
def col(df, clave):
    return next(c for c in df.columns if clave.lower() in c.lower())

c_mun  = col(df, "municipio")
c_dane = col(df, "dane")
c_mat  = col(df, "matr")
c_zona = col(df, "zona")
c_tec  = col(df, "tecnolog")
c_bw   = col(df, "ancho")
c_meses = col(df, "meses")

df[c_mat] = pd.to_numeric(df[c_mat], errors="coerce")
df[c_meses] = pd.to_numeric(df[c_meses], errors="coerce")
df[[c_mun, c_zona, c_mat]].describe(include="all")

,municipio,zona_urbana_rural,matr_cula_a_beneficiar_por
count,254,254,254.000000
unique,38,2,NaN
top,OCAÑA,URBANA,NaN
freq,34,186,NaN
mean,NaN,NaN,375.759843
std,NaN,NaN,388.331387
min,NaN,NaN,7.000000
25%,NaN,NaN,109.750000
50%,NaN,NaN,244.000000
75%,NaN,NaN,521.250000


## 3. Unidad de análisis

**Una fila = una sede educativa** (no una institución completa, ni un estudiante, ni un contrato). Lo respalda el código DANE de sede: es único en las 254 filas. La unidad es la sede porque el servicio de conectividad se asigna sede por sede, y cada sede reporta su propia matrícula a beneficiar y su zona.

In [6]:
print("Filas:", len(df))
print("Códigos DANE de sede únicos:", df[c_dane].nunique(), "->", "es llave única" if df[c_dane].is_unique else "HAY REPETIDOS")
print("Municipios distintos:", df[c_mun].nunique())

Filas: 254
Códigos DANE de sede únicos: 254 -> es llave única
Municipios distintos: 38


## 4. Pregunta analítica y variables clave

**Pregunta:** ¿Cómo se distribuye la matrícula beneficiada por el servicio de conectividad entre sedes urbanas y rurales, y en qué municipios se concentra?

**Variables clave:**
1. `ZONA (URBANA/RURAL)`: segmenta las sedes.
2. `MATRÍCULA A BENEFICIAR POR SEDE EDUCATIVA`: tamaño de la población que se beneficia.
3. `MUNICIPIO`: ubicación geográfica de la sede.

**Control de calidad previo:** en el CSV exportado, 19 valores de matrícula aparecen con decimales (de 1.075 a 2.152). Un número de estudiantes no puede ser decimal; lo más probable es un separador de miles mal interpretado (1.075 sería 1075). Es una hipótesis, hay que confirmarla con el portal; si la API trae valores enteros, esta corrección no se activa.

In [7]:
# Corrección condicional del posible separador de miles
no_enteros = df[df[c_mat] % 1 != 0]
print("Matrículas con decimales:", len(no_enteros))
df["matricula"] = np.where(df[c_mat] % 1 != 0, (df[c_mat] * 1000).round(), df[c_mat])
print("Rango corregido:", df["matricula"].min(), "-", df["matricula"].max())

Matrículas con decimales: 0
Rango corregido: 7 - 2152


In [8]:
resumen_zona = df.groupby(c_zona)["matricula"].agg(sedes="count", matricula_total="sum", promedio="mean", mediana="median").round(1)
resumen_zona["% de sedes"] = (resumen_zona["sedes"] / resumen_zona["sedes"].sum() * 100).round(1)
resumen_zona["% de matrícula"] = (resumen_zona["matricula_total"] / resumen_zona["matricula_total"].sum() * 100).round(1)
resumen_zona

,sedes,matricula_total,promedio,mediana,% de sedes,% de matrícula
zona_urbana_rural,,,,,,
RURAL,68,5598,82.3,46.5,26.8,5.9
URBANA,186,89845,483.0,356.5,73.2,94.1


In [9]:
df.groupby(c_mun)["matricula"].agg(sedes="count", matricula_total="sum").sort_values("matricula_total", ascending=False).head(10)

,sedes,matricula_total
municipio,,
OCAÑA,34,17312
VILLA DEL ROSARIO,24,16044
LOS PATIOS,18,10876
PAMPLONA,23,6973
TIBÚ,10,5040
ABREGO,12,4450
EL ZULIA,9,3650
EL TARRA,5,3277
CHINÁCOTA,6,2817


**Lectura con los datos del CSV (tras la corrección):** hay 254 sedes y 38 municipios. Las sedes rurales son 68 (26.8 % de las sedes) pero concentran solo 5,9 % de la matrícula a beneficiar (5.598 de 95.443 estudiantes); la mediana rural es de 46,5 estudiantes por sede frente a 356,5 en las urbanas. Ocaña (17.312), Villa del Rosario (16.044) y Los Patios (10.876) encabezan la matrícula. Esto describe cómo se reparte el servicio, no si es suficiente.

## 5. Riesgos y limitaciones del dataset

1. **Las variables de calidad no varían.** En las 254 sedes la tecnología es `Radio`, el ancho de banda es `20Mbps`, el tipo de servicio es `CONECTIVIDAD EN SEDE`, y el servicio dura 4 meses con las mismas fechas. Sin variabilidad no se puede comparar calidad entre sedes: el dataset sirve para ver *cobertura y tamaño*, no velocidad ni tecnología.
2. **Es un contrato puntual, no el estado actual.** El servicio va del 1 de julio al 30 de noviembre de 2022 y la consulta es de 2026. Además solo incluye las sedes beneficiadas por ese servicio, no necesariamente todas las sedes del departamento, así que no permite hablar de cobertura total.
3. **Errores de formato en matrícula.** 19 valores con decimales por un probable separador de miles; sin corregirlos, la matrícula total se subestima (68.605 frente a 95.443).
4. **Es una cifra planeada, no de uso.** "Matrícula a beneficiar" es la población objetivo; no dice cuántos estudiantes usaron el servicio ni si hubo interrupciones.
5. **`ZONA` no está definida en el dataset** (¿con qué criterio se clasifica como rural?), lo que limita la comparación urbano-rural.